In [ ]:
using Pkg 

In [ ]:
# Pkg.add("JuMP")
# Pkg.add("Gurobi")

In [ ]:
using JuMP, Gurobi
using Plots

## A first example
Let's see how we translate the tables-and-chairs example we just saw into Julia code.

$$
\begin{align*}
\max_{x_1,x_2} \quad
& 20 x_1 + 30 x_2 
\\
\text{such that} \quad
& x_1 + 2 x_2 \leq 30 \\
& 10 x_1 + 5 x_2 \leq 120 \\
& x_1, x_2 \geq 0.
\end{align*}
$$

We construct a model object. This is a container for everything in our optimization problem: variables, constraints, solver options, etc.

Note that when we specify the solver we can also pass additional arguments (depending on the solver we choose). 
If you're experimenting with an optimization model, it's generally a good idea to set a short time limit for the solver calculations. This will (hopefully) force the solver to stop running if you accidentally pass it a very large model that it can't solve. 

In [ ]:
model = Model(Gurobi.Optimizer)
set_optimizer_attribute(model, "TimeLimit", 60)

Next, we define the two decision variables in our optimization problem. We will use the ``@variable`` macro (a fancy function, essentially). The first argument is the model object to attach the variable to, and the second specifies the variable name and any bounds.

In [ ]:
@variable(model, x[1:2] ≥ 0)

In [ ]:
model

We now add the constraints of our problem using the ``@constraint`` macro:

In [ ]:
# Labor constraint
@constraint(model, 1 * x[1] + 2 * x[2] <= 30)
# Wood constraint
@constraint(model, 10 * x[1] + 5 * x[2] <= 120)
;

In [ ]:
model

We specify the objective function with the ``@objective`` macro.

In [ ]:
@objective(model, Max, 20 * x[1] + 30 * x[2])

In [ ]:
model

To solve the optimization problem, call the ``optimize!`` function.

In [ ]:
optimize!(model)
termination_status(model)

We can now inspect the solution values and optimal cost.

In [ ]:
x_sol = JuMP.value.(x)
x_sol

In [ ]:
JuMP.objective_value(model)

This means that one should build 6 tables (the value of $x_1$) and 12 chairs (the value of $x_2$) in order to maximize profit. This would result in $480 of profit.

We can also visualize the feasible region and optimal solution via `Plots.jl`. Here is a worked example:

In [ ]:
xmax = 30
ymax = 30
Plots.plot(
    xlabel="x_1 (Tables)", 
    ylabel="x_2 (Chairs)",  
    legend=:topright,
    aspect_ratio=:equal,
    xlims=(0, xmax),
    ylims=(0, ymax),
)
feasible_region = Shape(
    [ 
        (0, 0),
        (12, 0),
        (6, 12),
        (0, 15),
        (0, 0)
    ]
)
Plots.plot!(
    feasible_region, 
    fillcolor=:gray,
    fillalpha=0.5,
    label=false,
)
Plots.plot!(
    [(x, (30-x)/2) for x in 0:0.1:xmax], 
    label="x + 2y = 30", 
    linewidth=3,
)
Plots.plot!(
    [(x, (120-10*x)/5) for x in 0:0.1:xmax], 
    label="10x + 5y = 120", 
    linewidth=3,
)
Plots.plot!(
    [(0, y) for y in 0:0.1:ymax], 
    label="x = 0", 
    linewidth=3,
)
Plots.plot!(
    [(x, 0) for x in 0:0.1:xmax], 
    label="y = 0", 
    linewidth=3,
)
Plots.scatter!(
    [x_sol[1]], 
    [x_sol[2]], 
    label="Optimal Solution", 
    color=:black, 
    markersize=5
)
Plots.annotate!(
    x_sol[1] + 3, 
    x_sol[2] + 1, 
    text(
        "($(string(x_sol[1])), $(string(x_sol[2])))",
        :black,
        8,
        :center,
    )
)


## Exercise 1

Code and solve the following optimization problem:

$$
\begin{align*}
\min_{x,y} \quad& 3x - y \\
\text{s.t.}\quad& x + 2y \geq 1 \\
& x \geq 0 \\
& 0 \leq y \leq 1.
\end{align*}
$$

## Exercise 2

Take the problem from exercise 1 and make two copies below.
* In the first copy, add a new constraint to make the problem **infeasible** (i.e., there are no values of x and y that satisfy all the constraints)
* In the second copy, change the constraints or the objective function to make the problem unbounded (i.e., the optimal solution is infinite)

Solve both versions of the problem and look at the ``termination_status`` to see if you have succeeded.

Hint: below is code that plots the feasible region of the model from Exercise 1:

In [ ]:
xmax = 1.5
ymax = 1.5
Plots.plot(
    xlabel="x", 
    ylabel="y",  
    legend=:topright,
    aspect_ratio=:equal,
    xlims=(0, xmax),
    ylims=(0, ymax),
)
feasible_region = Shape(
    [ 
        (0, 0.5),
        (1, 0),
        (1.5, 0),
        (1.5, 1),
        (0, 1),
    ]
)
Plots.plot!(
    feasible_region, 
    fillcolor=:gray,
    fillalpha=0.5,
    label=false,
)
Plots.plot!(
    [(x, (1-x)/2) for x in 0:0.1:xmax], 
    label="x + 2y = 1", 
    linewidth=3,
)
Plots.plot!(
    [(x, 0) for x in 0:0.1:xmax], 
    label="y = 0", 
    linewidth=3,
)
Plots.plot!(
    [(x, 1) for x in 0:0.1:xmax], 
    label="y = 1", 
    linewidth=3,
)
Plots.plot!(
    [(0, y) for y in 0:0.1:ymax], 
    label="x = 0", 
    linewidth=3,
)

## Helpful resources

Here are some helpful links for your reference:
- A introduction to Julia on the JuMP website: [(link)](https://jump.dev/JuMP.jl/stable/tutorials/getting_started/getting_started_with_julia/)
- [The JuMP tutorial](https://jump.dev/JuMP.jl/stable/tutorials/getting_started/getting_started_with_JuMP/)
- the JuMP manual ([models](https://jump.dev/JuMP.jl/stable/manual/models/), [variables](https://jump.dev/JuMP.jl/stable/manual/variables/) and [constraints](https://jump.dev/JuMP.jl/stable/manual/constraints/))
- in the Gurobi documentation, [guidelines](https://www.gurobi.com/documentation/11.0/refman/mip_models.html) for parameter settings for MIP models (come back to this later)
- Packages to know in Julia:
    - [`DataFrames.jl`](https://dataframes.juliadata.org/stable/) for working with dataframes
    - [`Plots.jl`](https://docs.juliaplots.org/latest/) for plotting (among a few options)
- A [list](https://docs.julialang.org/en/v1/manual/noteworthy-differences/) of differences between Julia and other languages